# Week 2 — CrossEncoder Reranking: Precision After Recall

**Learning objectives**
- Understand the two-stage retrieval paradigm
- Know why cross-encoders are more accurate but slower than bi-encoders
- Load and use `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Measure MRR@5 improvement from reranking on the golden dataset

---

## 1. The Two-Stage Retrieval Paradigm

Modern production RAG systems split retrieval into two stages:

```
Query
  │
  ▼
Stage 1 — RECALL (fast, imprecise)
  BM25 top-20  +  Dense top-20  ──── RRF ───▶  40 candidates
  │
  ▼
Stage 2 — PRECISION (slow, accurate)
  CrossEncoder scores each (query, candidate) pair
  │
  ▼
Final top-5 for LLM context
```

**Why two stages?**  A cross-encoder is O(n) per query — running it over 100k documents would be prohibitively slow. Instead we use the fast bi-encoder to recall a small candidate pool (20–50 docs), then apply the expensive cross-encoder to just those candidates.

## 2. Bi-Encoder vs Cross-Encoder

| Property | Bi-Encoder | Cross-Encoder |
|---|---|---|
| Input | Query and document encoded **separately** | Query and document fed **together** |
| Interaction | Dot product / cosine of embeddings | Full attention across both sequences |
| Accuracy | Good (recall-oriented) | Excellent (precision-oriented) |
| Latency | Fast — embed corpus offline | Slow — must run per query-doc pair |
| Scalability | Scales to millions of docs | Only practical for tens of candidates |
| Example model | BAAI/bge-small-en-v1.5 | cross-encoder/ms-marco-MiniLM-L-6-v2 |

**Why is cross-encoder more accurate?**  In a bi-encoder, the query and document embeddings never directly interact — only their dot product does. A cross-encoder's transformer attention can attend query tokens to document tokens, capturing nuanced relevance signals that bi-encoders miss.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
from pathlib import Path

# Load golden dataset
with open("../evaluation/golden_dataset.json") as f:
    golden_dataset = json.load(f)

# Build mini corpus from all unique contexts
pension_corpus = []
seen = set()
for item in golden_dataset:
    for ctx in item["contexts"]:
        if ctx not in seen:
            pension_corpus.append(ctx)
            seen.add(ctx)

# Add some distractor passages to make the task harder
distractors = [
    "The board of trustees meets quarterly to review investment performance.",
    "Members may transfer their accrued pension rights to another provider.",
    "Currency hedging strategies must be documented in the investment policy.",
    "Derivative instruments may be used for efficient portfolio management only.",
    "The fund administrator must maintain records for a minimum of seven years.",
    "Actuarial valuations shall be conducted by a qualified actuary every three years.",
    "All investment decisions require sign-off from the Risk Committee.",
    "Performance benchmarks are reviewed annually by the Investment Committee.",
]
pension_corpus.extend(distractors)

print(f"Extended corpus: {len(pension_corpus)} passages ({len(distractors)} distractors added)")

## 3. Loading a CrossEncoder

`cross-encoder/ms-marco-MiniLM-L-6-v2` was fine-tuned on the MS MARCO passage ranking dataset — 530k real queries from Bing search. It outputs a single logit that can be interpreted as a relevance score.

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading CrossEncoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("CrossEncoder ready.")

# Quick sanity check — score a relevant and irrelevant passage
test_query = "What is the maximum recovery period in a pension fund recovery plan?"

relevant_passage = (
    "Article 14(2): The recovery plan must demonstrate restoration of the funding ratio "
    "to the required level within a period not exceeding ten years, using realistic "
    "actuarial assumptions."
)
irrelevant_passage = (
    "Article 31(1): Pension funds shall integrate ESG factors into their investment "
    "decision-making process."
)

scores = cross_encoder.predict([
    [test_query, relevant_passage],
    [test_query, irrelevant_passage],
])

print(f"\nQuery: {test_query!r}")
print(f"  Relevant passage score:   {scores[0]:.4f}")
print(f"  Irrelevant passage score: {scores[1]:.4f}")
print(f"\n  Score gap: {scores[0] - scores[1]:.4f} (higher = cross-encoder correctly distinguishes them)")

## 4. Score Distribution: Relevant vs Non-Relevant Chunks

In [ ]:
import matplotlib.pyplot as plt

# Pick a sample question
sample = golden_dataset[1]  # Article 14(2) recovery plan question
query = sample["question"]
gt_contexts = set(sample["contexts"])

# Score all corpus passages
pairs = [[query, doc] for doc in pension_corpus]
all_scores = cross_encoder.predict(pairs)

relevant_scores = [
    score for score, doc in zip(all_scores, pension_corpus)
    if doc in gt_contexts
]
non_relevant_scores = [
    score for score, doc in zip(all_scores, pension_corpus)
    if doc not in gt_contexts
]

print(f"Query: {query!r}")
print(f"Relevant passages: {len(relevant_scores)}, Non-relevant: {len(non_relevant_scores)}")
print(f"\nRelevant score mean:     {np.mean(relevant_scores):.3f}")
print(f"Non-relevant score mean: {np.mean(non_relevant_scores):.3f}")
print(f"Separation (mean gap):   {np.mean(relevant_scores) - np.mean(non_relevant_scores):.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(non_relevant_scores, bins=15, alpha=0.6, label="Non-relevant", color="steelblue")
ax.hist(relevant_scores,     bins=5,  alpha=0.8, label="Relevant",     color="darkorange")
ax.set_xlabel("CrossEncoder score")
ax.set_ylabel("Count")
ax.set_title("CrossEncoder Score Distribution: Relevant vs Non-Relevant")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Integrating Reranking into HybridRetriever

The `_rerank` method in `HybridRetriever` is called after RRF fusion. Here's a walkthrough of how it works.

In [ ]:
from rag_pipeline.retriever import HybridRetriever
from rag_pipeline.config import RAGConfig
from rag_pipeline.indexer import Indexer

# Build the full pipeline with reranking enabled
config = RAGConfig(
    chunking_strategy="recursive",
    chunk_size=400,
    chunk_overlap=40,
    initial_retrieval_k=15,
    final_top_k=5,
    use_bm25=True,
    use_reranking=True,
    reranking_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    chroma_persist_dir="./chroma_week2",
)

indexer = Indexer(config)
docs = [{"page_content": ctx, "metadata": {"source": "regulation"}}
        for ctx in pension_corpus]

print("Building indexes...")
chroma_collection = indexer.build_index(docs)
bm25_index = indexer.build_bm25_index(docs)
chunks_text = [d["page_content"] for d in docs]

retriever = HybridRetriever(
    config=config,
    chroma_store=chroma_collection,
    bm25_index=bm25_index,
    chunks_text=chunks_text,
)
print("Pipeline ready.")

In [ ]:
# Illustrate the _rerank method step-by-step
import inspect
print(inspect.getsource(HybridRetriever._rerank))

In [ ]:
# Show reranking in action: compare ordering before and after
test_query_2 = "What notification deadline applies when coverage ratio falls below 104.3%?"

# Stage 1 only (no reranking)
config_no_rerank = RAGConfig(
    chunking_strategy="recursive",
    chunk_size=400,
    chunk_overlap=40,
    initial_retrieval_k=15,
    final_top_k=5,
    use_bm25=True,
    use_reranking=False,
    chroma_persist_dir="./chroma_week2",
)
retriever_no_rerank = HybridRetriever(
    config=config_no_rerank,
    chroma_store=chroma_collection,
    bm25_index=bm25_index,
    chunks_text=chunks_text,
)

results_before = retriever_no_rerank.retrieve(test_query_2)
results_after  = retriever.retrieve(test_query_2)

print(f"Query: {test_query_2!r}\n")
print("BEFORE reranking (RRF only):")
for i, doc in enumerate(results_before):
    print(f"  [{i+1}] {doc['page_content'][:90]}...")

print("\nAFTER reranking (CrossEncoder):")
for i, doc in enumerate(results_after):
    score = doc['metadata'].get('rerank_score', 'N/A')
    print(f"  [{i+1}] score={score:.3f}  {doc['page_content'][:90]}...")

## 6. Latency vs Quality Trade-off

In [ ]:
import time

sample_query = "What is the minimum coverage ratio before surplus distribution?"
N_TRIALS = 10

# Bi-encoder only (no reranking)
latencies_no_rerank = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    retriever_no_rerank.retrieve(sample_query)
    latencies_no_rerank.append(time.perf_counter() - t0)

# With cross-encoder reranking
latencies_rerank = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    retriever.retrieve(sample_query)
    latencies_rerank.append(time.perf_counter() - t0)

print(f"Retrieval latency over {N_TRIALS} trials\n")
print(f"  Hybrid (no rerank):  mean={np.mean(latencies_no_rerank)*1000:.1f}ms  "
      f"p99={np.percentile(latencies_no_rerank, 99)*1000:.1f}ms")
print(f"  Hybrid + reranking:  mean={np.mean(latencies_rerank)*1000:.1f}ms  "
      f"p99={np.percentile(latencies_rerank, 99)*1000:.1f}ms")
print(f"\nOverhead: {(np.mean(latencies_rerank)/np.mean(latencies_no_rerank)):.1f}x")

In [ ]:
# Production strategies to mitigate reranking latency
production_strategies = {
    "Smaller reranking model": [
        "cross-encoder/ms-marco-MiniLM-L-2-v2 (2-layer, ~2x faster)",
        "Trade 1-3% MRR for 50% latency reduction",
    ],
    "Reduce candidate pool": [
        "Retrieve top-10 instead of top-20 before reranking",
        "Each additional candidate adds ~5ms on CPU",
    ],
    "Async / batching": [
        "Run reranking in background, stream bi-encoder results to UI immediately",
        "Use GPU batching for throughput-critical workloads",
    ],
    "Cache reranked results": [
        "Hash (query, candidate_set) → cache rerank scores",
        "Effective for repeated/similar queries (e.g. FAQ chatbots)",
    ],
    "ColBERT (late interaction)": [
        "Precompute per-token document embeddings; query-time MaxSim is fast",
        "Better quality/latency tradeoff than full cross-encoder for large corpora",
    ],
}

for strategy, details in production_strategies.items():
    print(f"• {strategy}")
    for d in details:
        print(f"    - {d}")
    print()

## 7. Benchmarking: MRR@5 Before and After Reranking

**Mean Reciprocal Rank (MRR@k)** = average of $\frac{1}{\text{rank}}$ of the first relevant result. It rewards finding the right answer at position 1 much more than at position 5.

In [ ]:
def mrr_at_k(results: list[dict], gt_contexts: set[str], k: int = 5) -> float:
    """Mean Reciprocal Rank for a single query."""
    for rank, doc in enumerate(results[:k], start=1):
        content = doc["page_content"]
        for gt in gt_contexts:
            if content[:80] in gt or gt[:80] in content:
                return 1.0 / rank
    return 0.0


mrr_no_rerank_scores = []
mrr_rerank_scores = []

for item in golden_dataset:
    query = item["question"]
    gt_set = set(item["contexts"])

    res_no_rerank = retriever_no_rerank.retrieve(query)
    res_rerank    = retriever.retrieve(query)

    mrr_no_rerank_scores.append(mrr_at_k(res_no_rerank, gt_set))
    mrr_rerank_scores.append(mrr_at_k(res_rerank, gt_set))

mrr_no_rerank = np.mean(mrr_no_rerank_scores)
mrr_rerank    = np.mean(mrr_rerank_scores)
improvement   = (mrr_rerank - mrr_no_rerank) / (mrr_no_rerank + 1e-9) * 100

print(f"Evaluation over {len(golden_dataset)} questions\n")
print(f"{'Method':<30}  {'MRR@5':>8}")
print("-" * 42)
print(f"{'Hybrid RRF (no reranking)':<30}  {mrr_no_rerank:>8.3f}")
print(f"{'Hybrid RRF + CrossEncoder':<30}  {mrr_rerank:>8.3f}")
print(f"{'Improvement':<30}  {improvement:>+7.1f}%")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: MRR comparison
methods = ["Hybrid\n(no reranking)", "Hybrid +\nCrossEncoder"]
mrr_values = [mrr_no_rerank, mrr_rerank]
colors = ["steelblue", "darkorange"]
ax1.bar(methods, mrr_values, color=colors, width=0.5)
ax1.set_ylabel("MRR@5")
ax1.set_title("MRR@5: Before vs After Reranking")
ax1.set_ylim(0, 1)
for i, v in enumerate(mrr_values):
    ax1.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

# Scatter: per-question gain
gains = [r - b for r, b in zip(mrr_rerank_scores, mrr_no_rerank_scores)]
ax2.scatter(range(len(gains)), gains,
            c=["green" if g > 0 else "red" if g < 0 else "gray" for g in gains],
            alpha=0.7, s=50)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_xlabel("Question index")
ax2.set_ylabel("MRR change (reranked − baseline)")
ax2.set_title("Per-Question MRR Change from Reranking")
n_improved = sum(1 for g in gains if g > 0)
n_degraded = sum(1 for g in gains if g < 0)
ax2.text(0.02, 0.95, f"Improved: {n_improved} | Degraded: {n_degraded}",
         transform=ax2.transAxes, va="top")

plt.tight_layout()
plt.show()

## 8. Key Takeaways + Exercises

### Takeaways

- **Two-stage retrieval** is the industry standard: fast recall (bi-encoder) + precise reranking (cross-encoder)
- Cross-encoders outperform bi-encoders because they model full query-document interactions via attention
- `ms-marco-MiniLM-L-6-v2` is an excellent lightweight cross-encoder: 6-layer MiniLM fine-tuned on 530k real queries
- Reranking typically adds 50–300ms on CPU — manageable for conversational latency targets
- The `rerank_score` is stored in document metadata for transparency and debugging

### Exercises

1. **Compare reranking models**: Try `cross-encoder/ms-marco-MiniLM-L-2-v2` (2-layer). Measure MRR@5 and latency vs the 6-layer model. What's the quality/speed trade-off?
2. **Score analysis**: For questions where reranking *degraded* MRR, inspect the CrossEncoder scores. Is the model systematically confused by certain topics?
3. **Threshold filtering**: Instead of always returning top-5, only return documents with `rerank_score > threshold`. What threshold maximises precision without reducing recall too much?
4. **Batch size**: The CrossEncoder's `predict()` call can take a batch. Experiment with `batch_size=8, 16, 32` and measure throughput improvement on GPU.